In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "cuckfonst/affine-exciter-3"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)


In [ ]:
model = AutoModelForCausalLM(model_name)



In [ ]:
messages = [
    {"role": "user", "content": "Hello, how are you?"},
    {"role": "assistant", "content": "I'm doing well, thanks!"},
    {"role": "user", "content": "What's the weather?"},
    # {"role": "assistant", "content": "I don't have access to weather data."}
]


In [ ]:
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,  # Switches between thinking and non-thinking modes. Default is True.
    # return_tensor="pt"
)
text

In [ ]:
tokens = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt")

In [ ]:
tokens.shape[1]
# len(tokens)

In [ ]:
text1 = tokenizer.apply_chat_template(
    [
        {"role": "user", "content": "Hello, how are you?"}
    ],
    tokenize=True,
    add_generation_prompt=False,
    enable_thinking=True,  # Switches between thinking and non-thinking modes. Default is True.
)

In [ ]:
text2 = tokenizer.apply_chat_template(
    [
        {"role": "assistant", "content": "I'm doing well, thanks!"}
    ],
    tokenize=True,
    add_generation_prompt=False,
    enable_thinking=True,  # Switches between thinking and non-thinking modes. Default is True.
)

In [ ]:
text1 + text2

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen3-0.6B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)


In [ ]:

# prepare the model input
prompt = "what is 27+46"
messages = [
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": "I don't know"},
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": "I don't know"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
)


In [ ]:

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

# conduct text completion
generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=32768
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 

# parsing thinking content
try:
    # rindex finding 151668 (</think>)
    index = len(output_ids) - output_ids[::-1].index(151668)
except ValueError:
    index = 0

thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

print("thinking content:", thinking_content)
print("content:", content)


In [ ]:
text

In [ ]:
tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("moonshotai/Kimi-K2.5")
tokenizer.eos_token_id

In [ ]:
# prepare the model input
prompt = "what is 27+46"
messages = [
    {"role": "user", "content": prompt},
    {"role": "assistant", "content": "I don't know"},
    {"role": "user", "content": prompt},
    # {"role": "assistant", "content": "I don't know", "reasoning_content": "REASONING PART"}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    thinking=True # Switches between thinking and non-thinking modes. Default is True.
)
print(text)

In [ ]:
messages = [{"content": "Review these logical expressions:\n\nA. not not (not (1995 is a leap year.))\nB. A duck has 2 legs.\nC. World War II began occurred in 1940.\n\nWhich expressions are true? Answer with comma-separated labels in \\boxed{}.", "role": "user"}, 

{"content": "\\boxed{A, B}", 

"reasoning_content": "We need to evaluate three logical expressions:\n\nA. not not (not (1995 is a leap year.))\nB. A duck has 2 legs.\nC. World War II began occurred in 1940.\n\nWe need to determine which are true.\n\nFirst, let's parse each.\n\nExpression A: \"not not (not (1995 is a leap year.))\"\n\nWe need to evaluate the inner statement: \"1995 is a leap year.\" Is 1995 a leap year? Leap years are divisible by 4, but not by 100 unless also by 400. 1995 mod 4 = 1995 - 4*498 = 1995 - 1992 = 3. So 1995 is not divisible by 4. So 1995 is not a leap year. So the statement \"1995 is a leap year\" is false.\n\nNow apply \"not\" to that: not (false) = true.\n\nThen apply \"not not\" to that: not not (true) = not (false) = true? Wait careful: The expression is \"not not (not (1995 is a leap year.))\". Let's parse parentheses:\n\nInner: (1995 is a leap year.) = false.\n\nThen not (false) = true.\n\nThen not not (true) = not (not (true)). Let's compute step by step:\n\nLet X = not (1995 is a leap year.) = not false = true.\n\nThen the outer is not not X = not (not X) = not (not true) = not false = true.\n\nAlternatively, not not Y = Y (double negation). So not not (not (1995 is a leap year.)) = not (1995 is a leap year.)? Wait, double negation cancels: not not Z = Z. So not not (not (1995 is a leap year.)) = not (1995 is a leap year.). Because Z = not (1995 is a leap year.). So not not Z = Z. So it's equivalent to \"not (1995 is a leap year.)\". Since 1995 is not a leap year, \"1995 is a leap year\" is false, so not false = true. So A is true.\n\nThus A is true.\n\nExpression B: \"A duck has 2 legs.\"\n\nIs that true? Ducks are birds, they have two legs. Yes, a typical duck has two legs. So B is true.\n\nExpression C: \"World War II began occurred in 1940.\"\n\nThis is a bit odd phrasing: \"began occurred\". Probably a typo: \"World War II began in 1940\" or \"World War II occurred in 1940\". But the statement says \"World War II began occurred in 1940.\" That seems ungrammatical. However, we need to interpret the logical content. It says \"World War II began occurred in 1940.\" This is likely a mistake, but we need to evaluate truth. World War II began in 1939 (September 1, 1939). So the statement that it began in 1940 is false. Also \"began occurred\" is nonsense, but the intended meaning is probably \"World War II began in 1940\" which is false. So C is false.\n\nThus true expressions are A and B.\n\nAnswer: \\boxed{A, B}\n\nBut we need to be careful: The question says \"Review these logical expressions: A. not not (not (1995 is a leap year.)) B. A duck has 2 legs. C. World War II began occurred in 1940. Which expressions are true? Answer with comma-separated labels in \\boxed{}.\"\n\nThus answer: \\boxed{A, B}\n\nBut let's double-check each.\n\nA: 1995 is not a leap year. So \"1995 is a leap year\" is false. \"not (1995 is a leap year)\" is true. \"not not (not (1995 is a leap year))\" = not not (true) = true. So A is true.\n\nB: \"A duck has 2 legs.\" This is a factual statement. Ducks have two legs. So true.\n\nC: \"World War II began occurred in 1940.\" This is false. WWII began in 1939. Also the phrase \"began occurred\" is contradictory or ungrammatical, making it false.\n\nThus answer: A, B.\n\nBut we need to consider if there is any trick. For B, is there any nuance? Some ducks might have lost a leg, but the statement \"A duck has 2 legs\" is a general statement about the species. In logic, it's a universal statement? It says \"A duck has 2 legs.\" This is ambiguous: It could be interpreted as \"All ducks have 2 legs\" or \"There exists a duck that has 2 legs.\" Usually, \"A duck has 2 legs\" is a generic statement meaning \"Ducks have 2 legs.\" That is true. Even if interpreted as \"There exists a duck that has 2 legs,\" it's also true. So B is true.\n\nC: \"World War II began occurred in 1940.\" The phrase \"began occurred\" is weird. Could it be a typo for \"World War II began or occurred in 1940\"? Or \"World War II began, occurred in 1940\"? The phrase \"began occurred\" is likely a mistake. But the truth value: WWII began in 1939, not 1940. So false.\n\nThus answer: A, B.\n\nBut let's consider if the phrase \"began occurred\" might be interpreted as \"began and occurred\"? That would be \"World War II began and occurred in 1940.\" That is false. Or \"began, occurred in 1940\"? Still false.\n\nThus answer: \\boxed{A, B}\n\nOne more nuance: The expression A uses \"not not\". In classical logic, double negation cancels. So not not (not P) = not P. So it's equivalent to \"1995 is not a leap year.\" That is true. So A is true.\n\nThus final answer: \\boxed{A, B}\n\nThe question says \"comma-separated labels\". So \"A, B\". Possibly they want \"A,B\" without space? But \"comma-separated\" usually allows spaces. The example: \"Answer with comma-separated labels in \\boxed{}.\" So we can put \\boxed{A, B} or \\boxed{A,B}. Both are fine. I'll include a space after comma for readability.\n\nThus final answer: \\boxed{A, B}.\n\nBut wait, check if any other interpretation could make C true. WWII began in 1939, but some might say the US entered in 1941, but the war began in 1939. The phrase \"began occurred in 1940\" is nonsense. Could it be \"World War II began, occurred in 1940\"? No. Could it be \"World War II began; occurred in 1940\"? No. The phrase \"began occurred\" is likely a typo for \"began or occurred\"? Or \"began, occurred\"? Or \"began in 1940\"? The most plausible is that it's a mistake and they meant \"World War II began in 1940.\" That is false. So C is false.\n\nThus answer: A, B.\n\nDouble-check: Is 1995 a leap year? 1995 is not divisible by 4. 1996 is a leap year. So 1995 is not. So not (1995 is leap) is true. So A is true.\n\nThus answer: \\boxed{A, B}.\n\nOne more nuance: The phrase \"not not (not (1995 is a leap year.))\" includes a period inside the parentheses. Does that matter? It's just punctuation.\n\nThus final answer: \\boxed{A, B}.\n\nBut let's consider if the question is a trick about the phrase \"began occurred\". Could it be that \"began\" and \"occurred\" are two separate predicates? Like \"World War II began\" and \"World War II occurred in 1940\"? The phrase is \"World War II began occurred in 1940.\" That is ungrammatical. It might be a copy-paste error. If we interpret it as \"World War II began in 1940\", false. If we interpret as \"World War II occurred in 1940\", also false (it lasted 1939-1945). So false.\n\nThus answer: A, B.\n\nThus I will output \\boxed{A, B}.", "role": "assistant"}]

In [ ]:
tokenizer.chat_template = r"""{%- macro render_content(msg) -%}
    {%- set c = msg.get('content') -%}
    {%- if c is string -%}
      {{ c }}
    {%- elif c is not none -%}
      {%- for content in c -%}
        {%- if content['type'] == 'image' or content['type'] == 'image_url' -%}
          <|media_begin|>image<|media_content|><|media_pad|><|media_end|>
        {%- elif content['type'] == 'video' or content['type'] == 'video_url' -%}
          <|kimi_k25_video_placeholder|>
        {%- else -%}
          {{ content['text'] }}
        {%- endif -%}
      {%- endfor -%}
    {%- endif -%}
  {%- endmacro -%}
  {%- macro set_roles(message) -%}
    {%- set role_name = message.get('name') or message['role'] -%}
    {%- if message['role'] == 'user' -%}
      <|im_user|>{{ role_name }}<|im_middle|>
    {%- elif message['role'] == 'assistant' -%}
      <|im_assistant|>{{ role_name }}<|im_middle|>
    {%- else -%}
      <|im_system|>{{ role_name }}<|im_middle|>
    {%- endif -%}
  {%- endmacro -%}
  {%- macro render_toolcalls(message) -%}
    <|tool_calls_section_begin|>
    {%- for tool_call in message['tool_calls'] -%}
      {%- set formatted_id = tool_call['id'] -%}
      <|tool_call_begin|>{{ formatted_id }}<|tool_call_argument_begin|>{% if tool_call['function']['arguments'] is string %}{{ tool_call['function']['arguments'] }}{% else %}{{ tool_call['function']['arguments'] | tojson }}{% endif %}<|tool_call_end|>
    {%- endfor -%}
    <|tool_calls_section_end|>
  {%- endmacro -%}
  {%- set ns = namespace(last_non_tool_call_assistant_msg=-1) -%}
  {%- for idx in range(messages|length-1, -1, -1) -%}
    {%- if messages[idx]['role'] == 'assistant' and not messages[idx].get('tool_calls') -%}
      {%- set ns.last_non_tool_call_assistant_msg = idx -%}
      {%- break -%}
    {%- endif -%}
  {%- endfor -%}
  {%- set hist_msgs = messages[:ns.last_non_tool_call_assistant_msg] -%}
  {%- set suffix_msgs = messages[ns.last_non_tool_call_assistant_msg:] -%}
  {%- if tools -%}
    {%- if tools_ts_str -%}
      <|im_system|>tool_declare<|im_middle|>{{ tools_ts_str }}<|im_end|>
    {%- else -%}
      <|im_system|>tool_declare<|im_middle|>{{ tools | tojson(separators=(',', ':')) }}<|im_end|>
    {%- endif -%}
  {%- endif -%}
  {%- for message in hist_msgs -%}
    {{ set_roles(message) }}
    {%- if message['role'] == 'assistant' -%}
      <think></think>{{ render_content(message) }}
      {%- if message.get('tool_calls') -%}
        {{ render_toolcalls(message) }}
      {%- endif -%}
    {%- elif message['role'] == 'tool' -%}
      {%- set tool_call_id = message.tool_call_id -%}
      ## Return of {{ tool_call_id }}
      {{ render_content(message) }}
    {%- elif message['content'] is not none -%}
      {{ render_content(message) }}
    {%- endif -%}
    <|im_end|>
  {%- endfor -%}
  {%- for message in suffix_msgs -%}
    {{ set_roles(message) }}
    {%- if message['role'] == 'assistant' -%}
      {%- if thinking is defined and thinking is false -%}
        <think></think>{{ render_content(message) }}
      {%- else -%}
        {%- set rc = message.get('reasoning_content', '') -%}
        <think>{{ rc }}</think>{{ render_content(message) }}
      {%- endif -%}
      {%- if message.get('tool_calls') -%}
        {{ render_toolcalls(message) }}
      {%- endif -%}
    {%- elif message['role'] == 'tool' -%}
      {%- set tool_call_id = message.tool_call_id -%}
      ## Return of {{ tool_call_id }}
      {{ render_content(message) }}
    {%- elif message['content'] is not none -%}
      {{ render_content(message) }}
    {%- endif -%}
    <|im_end|>
  {%- endfor -%}
  {%- if add_generation_prompt -%}
    <|im_assistant|>assistant<|im_middle|>
    {%- if thinking is defined and thinking is false -%}
      <think></think>
    {%- else -%}
      <think>
    {%- endif -%}
  {%- endif -%}"""

In [ ]:
print(tokenizer.chat_template)

In [ ]:
tokenizer.save_pretrained("./tokenizer")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "moonshotai/Kimi-K2.5",
    trust_remote_code=True,
)

# 2. Load base INT4 model
base_model = AutoModelForCausalLM.from_pretrained(
    "moonshotai/Kimi-K2.5",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map="auto",  # Distributes across your 8x H100
)

# 3. Load your trained LoRA adapters
model = PeftModel.from_pretrained(
    base_model,
    "checkpoints/kimi_k25_sft/model/",  # Path to your adapter checkpoint
    torch_dtype=torch.bfloat16,
)

# 4. Set to evaluation mode
model.eval()

# 5. Test inference
prompt = "What's the best opening move in chess?"

messages = [
    {"role": "user", "content": prompt}
]

# Apply chat template
input_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=2000,
        temperature=0.7,
        do_sample=True,
    )

# Decode and print
response = tokenizer.decode(outputs[0], skip_special_tokens=False)
print(response)